# Agendapunten & Besluitvorming

**Thema:** Hoe agendapunten leiden tot besluiten binnen vergaderingen.

**Kernvragen:** Hoe lang duren agendapunten gemiddeld? Welke soort besluiten (aangenomen, verworpen, aangehouden) komen het meeste voor per soort zaak?

Dit notebook verkent o.a. de silver-tabellen `agendapunt`, `besluit`, `besluit_zaak`, `zaak_agendapunt` en `document_agendapunt`.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

## Inladen gebruikte silver datatabellen

In [ ]:
# Schrijf hier je Python/SQL code
agendapunt = query_df("""
    SELECT *
    FROM silver.agendapunt
""")

besluit = query_df("""
    SELECT *
    FROM silver.besluit
""")

besluit_zaak = query_df("""
    SELECT *
    FROM silver.besluit_zaak
""")

stemming = query_df("""
    SELECT *
    FROM silver.stemming
""")

activiteit = query_df("""
    SELECT *
    FROM silver.activiteit
""")

zaak = query_df("""
    SELECT *
    FROM silver.zaak
""")

zaak_actor = query_df("""
    SELECT *
    FROM silver.zaak_actor
""")

fractie = query_df("""
    SELECT *
    FROM silver.fractie
""")

# 1. Duraties
In dit stuk zullen vragen omtrend de duratie van agendapunten/activiteiten beantwoord worden. Onderwerpen zoals de gemiddelde duratie, spreiding van de duraties, en invloed van de duratie op stemgedrag worden behandeld. 

## 1.1 Omtrendt agendapunten

### 1.1.1 Hoe lang duren agendapunten gemiddeld?
Eerst belangrijk voor deze vraag is om te kijken hoeveel van de agendapunten een bekende duratie hebben. Aan de hand daarvan zal een grafiek met de verschillende duraties en het gemiddelde gemaakt worden. 

In [ ]:
# Aantal agendapunten met een bekende aanvangst en eindtijd
agendapunten_met_tijden = agendapunt[
    agendapunt["aanvangstijd"].notna()
    & agendapunt["eindtijd"].notna()]

# Duratie agendapunten berekenen
agendapunt["duur_minuten"] = (
    agendapunt["eindtijd"] - agendapunt["aanvangstijd"]
).dt.total_seconds() / 60

print(f"Percentage agendapunten met duratie: {round(len(agendapunten_met_tijden) / len(agendapunt) * 100, 2)}%")
print(f"Datatype: {agendapunt["duur_minuten"].dtype}")
print(f"Aantal unique tijden: {agendapunt["duur_minuten"].unique()}")

Eerste opvallende is dat een zeer klein percentage van de agendapunten een bekende duratie hebben. De relevantie van de resultaten omtrend de duratie van agendapunten is dus betwijfelbaar. Desondanks zullen ze wel uitgewerkt worden.

Er zijn verbazingwekkend weinig verschillende soorten duraties. Ook zijn ze allen een veelvoud van 5. Door de weinige tijden, kan dit resultaat regelrecht in een grafiek weergegeven worden. 

In [ ]:
agendapunt["duur_minuten"] = agendapunt["duur_minuten"].astype("Int64")

gemiddelde_tijd_agendapunt = agendapunt["duur_minuten"].mean()

agendapunt["duur_minuten"].value_counts().sort_index().plot.bar(
    xlabel="Duur (minuten)",
    ylabel="Aantal agendapunten",
    title="Aantal agendapunten per tijdduur")

print(f"Gemiddelde duratie agendapunt: {round(gemiddelde_tijd_agendapunt, 2)} minuten.")
print(f"Mediaan duratie agendapunt: {agendapunt["duur_minuten"].median()} minuten")

De spreiding van de tijden is vrij groot, met een maximum van 780 minuten = 13 uur. Dit is uitzonderlijk lang, en samen met de pieken rond 0, 60, 120 en 180 minuten is de nauwkeirigheid en juisthied van de data zeer in twijfel te trekken. De gemiddelde duratie van een agendapunt ligt op 83 minuten, terwijl de mediaan bij 60 minuten ligt. 

### 1.1.2 Wat is het langste agendapunt?
Dit is duidelijk te zien in de grafiek hierboven. Het langste agendapunt waarvan de duratie bekend is duurde 780 minuten = 13 uur.

### 1.1.3 Heeft de duratie van een agendapunt invloed op het stemgedrag?
De tijd die aan agendapunten wordt gegeven varieert sterk. Hier wordt gekenen of dit invloed kan hebben op het stemgedrag, bijvoorbeeld agendapunten met bepaalde duraties die grotere marges (overweldigend voor/tegen/niet deelgenomen) hebben., 

In [ ]:
# df voor soort stem per duratie
aantal_stemmen_duratie = query_df("""
SELECT
    a.aanvangstijd,
    a.eindtijd,
    b.agendapunt_id,
    s.soort
FROM agendapunt a
JOIN besluit b
    ON a.id = b.agendapunt_id
JOIN stemming s
    ON b.id = s.besluit_id
""")

# Duratie van agendapunten berekenen
aantal_stemmen_duratie["duur_minuten"] = (
    aantal_stemmen_duratie["eindtijd"] - aantal_stemmen_duratie["aanvangstijd"]
).dt.total_seconds() / 60

aantal_stemmen_duratie['duur_minuten'].unique()

Bovenstaand dataframe is leeg. Er zijn geen agendapunten met duraties waarvan er ook gegevens over het stemgedrag beschikbaar zijn. Hieronder is onderzocht waar de gegevens verloren gaan.

In [ ]:
tijden_met_besluit = agendapunten_met_tijden[agendapunten_met_tijden["id"].isin(besluit["agendapunt_id"])]

tijden_besluit_id = besluit[besluit["agendapunt_id"].isin(tijden_met_besluit["id"])]

tijden_besluit_stemming = tijden_besluit_id[tijden_besluit_id["id"].isin(stemming["besluit_id"])]

print("Aantal agendapunten met tijden:", len(agendapunten_met_tijden))
print("Aantal agendapunten met besluiten:", len(tijden_met_besluit))
print("Aantal besluit bij alle agendapunten:", len(tijden_besluit_id))
print("Aantal tijdgebonden besluiten met stemgegevens:", len(tijden_besluit_stemming))

Het aantal agendapunten met tijden is 0.06% van de totale data die beschikbaar is. Van deze agendapunten zijn er maar 44 met overeenkomende besluiten. Omdat sommige agendapunten meerdere besluiten hebben, word het aantal besluiten waarvan we tijdgegevens hebben iets groter met 82. Echter hebben geen van deze besluiten overeenkomende stemgegevens. 

Met de beschikbare data is er dus geen antwoord te geven op de invloed van de duratie van een agendapunt op het stemgedrag. 

## 1.2 Omtrendt activiteiten

In [ ]:
# Dataframe dat gebruikt wordt voor alle vragen in 1.2
activiteit_besluit = query_df("""
    SELECT 
        a.id AS activiteit_id,
        a.datum_soort,
        a.aanvangstijd,
        a.eindtijd,
        b.besluit_soort,
        b.besluit_tekst
    FROM activiteit a
    JOIN agendapunt ap
        ON a.id = ap.activiteit_id
    JOIN besluit b
        ON ap.id = b.agendapunt_id
""")

### 1.2.1 Hoelang duren activiteiten gemiddeld en wat is de spreiding?

In [ ]:
# Duratie van de activiteiten berekenen. 
activiteit_besluit["duur_minuten"] = (
    activiteit_besluit["eindtijd"]
    - activiteit_besluit["aanvangstijd"]
).dt.total_seconds() / 60

Bovenstaande berekening van de duratie van activiteiten geeft een aantal negatieve waarden terug. Dit komt door foutieve invoer bij de begin- of eindtijden. Een klein voorbeeld van de fouten in de dataframe is zichtbaar hieronder. Hierna worden de negatieve duraties weggehaald uit de analyse. Ook wordt er gekozen om alleen naar activiteiten te kijken die ééndaags zijn, omdat er anders teveel extreme waarden zijn.

In [ ]:
activiteit_besluit.loc[
    activiteit_besluit["duur_minuten"] < 0,
    ["activiteit_id", "aanvangstijd", "eindtijd", "duur_minuten"]
].head(3)

In [ ]:
# Negatieve duraties weghalen
activiteit_besluit = activiteit_besluit[
    activiteit_besluit["duur_minuten"] >= 0
].copy()

In [ ]:
# Alleen de duratie van de activiteiten die één dag duren
activiteit_besluit_dag = activiteit_besluit.loc[
    activiteit_besluit["datum_soort"] == "Dag",
    "duur_minuten"]

In [ ]:
# Percentage 0 minuten
percentage_zero = (activiteit_besluit_dag == 0).sum() / len(activiteit_besluit_dag) *100
print(f"{percentage_zero:.2f}%")

# Verdeling van de duratie van de ééndaagse activiteiten
activiteit_besluit_dag.describe()

Ditmaal zijn er genoeg datapunten overgebleven om een zinnige analyse te kunnen uitvoeren. Ook is al duidelijk te zien dat de data zeer scheef verdeeld is, met 75% van de duraties onder 240 minuten, maar met een maximum van 1439 minuten (=23.99 uur).

In [ ]:
activiteit_besluit_dag.plot.hist(
    bins=50,
    xlabel="Duur (minuten)",
    ylabel="Frequentie",
    title="Verdeling van activiteitenduur"
)

In [ ]:
upper_bound = activiteit_besluit_dag.quantile(0.95)

activiteit_besluit_dag.loc[activiteit_besluit_dag <= upper_bound].plot.hist(
    bins=50,
    xlabel="Duur (minuten)",
    ylabel="Frequentie",
    title="Verdeling van activiteitenduur"
)

Het is duidelijk dat de meeste ééndaagse activiteiten rond de 2 uur duren, met een gemiddelde van ongeveer 1.5 uur. Ecter zijn er ook een significant aantal activiteiten die langer duren: 95% van de activiteiten duren ongeveer onder de 10 uur. De maximale duratie die gevonden is bedraagt 23.99 uur.  

### 1.2.2 Heeft de duratie van een activiteit invloed op de besluiten?
Hierbij kijken we nogmaals alleen naar de ééndaagse activiteiten. 

In [ ]:
# Alleen de ééndaagse activiteiten
activiteiten_besluit = activiteit_besluit[
    activiteit_besluit["datum_soort"] == "Dag"
].copy()

In [ ]:
# Vindt de besluiten via stemming met een van de drie categorieën
# Zet alleen de categorie in een aparte kolom voor makelijke analyse
activiteit_besluit["besluit_kort"] = activiteit_besluit["besluit_soort"].str.extract(
    r"(aangenomen|verworpen|aangehouden)",
    expand=False)

In [ ]:
# Alle actieviteiten met besluit en duratie
invloed_duratie_besluit_activiteit = activiteit_besluit[
    activiteit_besluit["besluit_kort"].notna()
    & activiteit_besluit["duur_minuten"].notna()].copy()

In [ ]:
# De hoeveelheid datapunten bij elke categorie
# Gemiddelde en mediaan van de duratie bij elke categorie
invloed_duratie_besluit_activiteit.groupby("besluit_kort")["duur_minuten"].agg(
    ["count", "mean", "median"]
).round(2)

In [ ]:
# Boxplot van de besluiten en de duratie van de activiteit
invloed_duratie_besluit_activiteit.boxplot(
    column="duur_minuten",
    by="besluit_kort"
)

plt.xlabel("Besluit")
plt.ylabel("Duur activiteit (minuten)")
plt.title("Duur van activiteit per besluituitkomst")
plt.suptitle("")
plt.show()

Bij alle drie de soorten besluiten is opvallend dat de mediale duratie veel lager ligt dan de gemiddelde duratie. Vanuit de boxplot is zeer opvallend dat de duratie van de activiteiten die aangenomen worden veel lager is dan de duraties bij aangehouden of verworpen. 

Vanuit deze analyse lijkt het erop dat de activiteiten met kortere duraties meer kans hebben om aangenomen te worden. 

# 2. Besluiten op zaken voorgelegdt door bepaalde fracties

## 2.1 Worden voorstellen van bepaalde fracties vaak aangenomen of verworpen (%)?

Hiermee kan gezien worden hoe succesvol/sterk de fractie en hun voorstellen zijn geweest over de jaren. Ook kan het gebruit worden om een algemene inschatting te maken van waar Nederland gemiddeld ligt op het politieke spectrum (links/rechts, progressief/conservatief). 

Omdat niet alle fracties meer actief zijn, is deze analyse 2 maal uitgevoerd: éénmaal voor alle fracties, ongeacht van hun status, en éénmaal voor de fracties die momenteel actief zijn.

Eerst worden tabellen gegeven waarin de percentages voor alle fracties staan (2.1.1 en 2.1.2), daarna worden tabellen gegeven waarin de top 3 staat per besluit (2.2.1 en 2.2.2).

### 2.1.1 Alle fracties (inclusief inactieve fracties)

In [ ]:
# Opstellen van dataframe met alle benodigde info
besluit_voorstel_per_fractie = query_df("""
    SELECT 
        z.id AS zaak_id,
        b.id AS besluit_id,
        b.besluit_soort,
        b.besluit_tekst,
        za.actor_fractie AS fractie_actor_zaak

    FROM zaak z
    JOIN besluit_zaak bz
        ON z.id = bz.zaak_id
    JOIN besluit b
        ON bz.besluit_id = b.id
    JOIN zaak_actor za
        ON z.id = za.zaak_id
""")

In [ ]:
# Toevoegen van aangehouden/verworpen/aangenomen voor makelijkere analyse
besluit_voorstel_per_fractie["besluit_kort"] = besluit_voorstel_per_fractie["besluit_soort"].str.extract(
    r"(aangenomen|verworpen|aangehouden)",
    expand=False)

# Aantal besluiten die over de zaken gemaakt zijn, per fractie
per_fractie = (
    besluit_voorstel_per_fractie[
        ["zaak_id", "fractie_actor_zaak", "besluit_kort"]
    ]
    .drop_duplicates()
    .dropna(subset=["besluit_kort", "fractie_actor_zaak"])
)

# Aantal zaken per fractie
aantal_zaken_per_fractie = (
    besluit_voorstel_per_fractie
    .dropna(subset=["fractie_actor_zaak"])
    .groupby("fractie_actor_zaak")["zaak_id"]
    .nunique()
    .rename("aantal_zaken")
)

# Berekenen van percentages
percentage_per_fractie = (
    per_fractie
    .groupby(["fractie_actor_zaak", "besluit_kort"])
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum() * 100)
    .reset_index(name="percentage")
)

percentage_per_fractie["percentage"] = (
    percentage_per_fractie["percentage"].round(2)
)

# Dataframe in leesbaardere vorm zetten
percentage_per_fractie_pivot = (
    percentage_per_fractie
    .pivot(
        index="fractie_actor_zaak",
        columns="besluit_kort",
        values="percentage"
    )
    .fillna(0)
    .round(2)
)

# Aantal zaken toevoegen
percentage_per_fractie_pivot = (
    percentage_per_fractie_pivot
    .join(aantal_zaken_per_fractie)
)

percentage_per_fractie_pivot

Hierboven zijn de resultaten voor alle fracties. Als er gezocht wordt naar een specifieke fractie kan dit tabel het beste gebruikt worden. Voor wie percentueel de meeste aangehouden/verworpen/aangenomen voorstellen heeft, zie 2.2.1

### 2.1.2 Alleen actieve fracties

In [ ]:
# Selecteerd alleen de actieve fracties
besluit_voorstel_per_actieve_fractie = query_df("""
    SELECT 
        z.id AS zaak_id,
        b.id AS besluit_id,
        b.besluit_soort,
        b.besluit_tekst,
        za.actor_fractie AS fractie_actor_zaak

    FROM zaak z
    JOIN besluit_zaak bz
        ON z.id = bz.zaak_id
    JOIN besluit b
        ON bz.besluit_id = b.id
    JOIN zaak_actor za
        ON z.id = za.zaak_id
    JOIN fractie f
        ON za.fractie_id = f.id

    WHERE f.datum_inactief IS NULL
""")

In [ ]:
# Toevoegen van aangehouden/verworpen/aangenomen voor makelijkere analyse
besluit_voorstel_per_actieve_fractie["besluit_kort"] = (
    besluit_voorstel_per_actieve_fractie["besluit_soort"]
    .str.extract(
        r"(aangenomen|verworpen|aangehouden)",
        expand=False
    )
)

# Aantal besluiten die over de zaken gemaakt zijn, per fractie
per_actieve_fractie = (
    besluit_voorstel_per_actieve_fractie[
        ["zaak_id", "fractie_actor_zaak", "besluit_kort"]
    ]
    .drop_duplicates()
    .dropna(subset=["besluit_kort"])
)

# Aantal unieke zaken per fractie
aantal_zaken_per_fractie = (
    per_actieve_fractie
    .groupby("fractie_actor_zaak")["zaak_id"]
    .nunique()
    .rename("aantal_zaken")
)

# Berekenen van percentages
percentage_per_actieve_fractie = (
    per_actieve_fractie
    .groupby(["fractie_actor_zaak", "besluit_kort"])
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum() * 100)
    .reset_index(name="percentage")
    .round(2)
)

# Dataframe in leesbaardere vorm zetten
percentage_per_actieve_fractie = (
    percentage_per_actieve_fractie
    .pivot(
        index="fractie_actor_zaak",
        columns="besluit_kort",
        values="percentage"
    )
    .fillna(0)
    .round(2)
)

# Aantal zaken toevoegen
percentage_per_actieve_fractie = (
    percentage_per_actieve_fractie
    .join(aantal_zaken_per_fractie)
)

percentage_per_actieve_fractie

Hierboven zijn de percentages voor alle fracties die op het moment actief zijn. Belangrijk om te noteren is dat de fractie 50PLUS niet voorkomt in dit tabel, omdat deze fractie sinds hun wederkeren (2025-22-12) nog geen zaken gestart heeft. De zaken die ze voorheen voorgedragen heben tellen niet mee, sinds hun wederkeer een nieuwe partij is, zelfs al hebben ze dezelfde naam. Voor wie percentueel de meeste aangehouden/verworpen/aangenomen voorstellen heeft, zie 2.2.2

## 2.2 Van welke fracties worden de meeste besluiten verworpen/aangenomen/aangehouden?

### 2.2.1 Alle fracties (inclusief inactieve fracties)
Bij het tabel met alle fracties is opvallend dat er twee fracties zijn met een 100% aangenomen: Ephraim en GrKH. Omdat er verdenking op kortstondige fracties (door lage aantal zaken), is er eerst gekeken naar wanneer en hoe lang deze fracties actief waren. 

In [ ]:
# Bij alle fracties zijn deze uitgesloten door 100% aangenomen
# Fracties hadden beide zeer korte levensduur
fractie.loc[
    fractie["afkorting"].isin(["Ephraim", "GrKH"]),
    ["afkorting", "datum_actief", "datum_inactief"]
]

Beide fracties zijn slechts een paar maanden actief geweest. Door deze korte bestaansduur en het lage aantal zaken, is er voor gekozen om deze fracties (fracties met een 100% in de categorieën) uit te sluiten bij de analyse.

In [ ]:
# Alle fracties top 3 per catagorie
top_3_per_categorie = (
    percentage_per_fractie_pivot
    .loc[
        ~percentage_per_fractie_pivot.eq(100).any(axis=1)
    ]
    .reset_index()
    .melt(
        id_vars=["fractie_actor_zaak", "aantal_zaken"],
        var_name="categorie",
        value_name="percentage"
    )
    .sort_values(
        ["categorie", "percentage"],
        ascending=[True, False]
    )
    .groupby("categorie")
    .head(3)
)

top_3_per_categorie = top_3_per_categorie[
    ["fractie_actor_zaak", "categorie", "percentage", "aantal_zaken"]
]

top_3_per_categorie

De percentages bij aangehouden zijn allen vrij laag. Dit is vrij logisch: weinig zaken zullen uitgesteld worden of geen definitieve uitkomst hebben. Voor aangenomen of verworpen liggen de percentages allen vrij dicht bij elkaar (rond 80%). 

Voor de partijen met het hoogste aangenomen percentage geldt dat ze allen redelijk bij elkaar liggen in het politieke spectrum. De vvd is de grootste uitbijter, met een duidelijke rechts en conservatieve voorkeur, terwijl nsc en cda beiden in het midden liggen met een lichte voorkeur naar conservatief.

Voor de partijen met de meeste verworpen zaken geldt dat ze ook allen rechts conservatief liggen in het politieke spectrum. Voor de GrBvK (Groep Bontes/Van Klaveren) en Brinkman kon geen echt duidelijk plaatje gevonden worden van hoe extreem rechts/conservatief deze waren. Voor de FVD geldt dat ze extreem rechts en conservatief zijn. 

Hoewel Nederlandse fracties over de jaren dus succes hebben gezien met rechts/conservatief, lijkt er een limiet te zijn voor hoe ver rechts/conservatief je kan zijn. 

### 2.2.2 Alleen actieve fracties

In [ ]:
top_3_per_categorie_actief = (
    percentage_per_actieve_fractie
    .reset_index()
    .melt(
        id_vars=["fractie_actor_zaak", "aantal_zaken"],
        var_name="categorie",
        value_name="percentage"
    )
    .sort_values(
        ["categorie", "percentage"],
        ascending=[True, False]
    )
    .groupby("categorie")
    .head(3)
)

top_3_per_categorie_actief = top_3_per_categorie_actief[
    ["fractie_actor_zaak", "categorie", "percentage", "aantal_zaken"]
]

top_3_per_categorie_actief

Ook hier zijn de percentages bij de aangehouden zaken vrij laag. De aangenomen en verworpen percentages zijn vele malen hoger, met een vrij grote spreiding. 

Bij de aangenomen categorie nemen de VVD en het CDA nogmaals het voortouw, maar is de derde fractie is veranderd naar PRO. Voor de ligging van de partijen geldt nogmaals rechts/conservatief voor het CDA en de VVD, maar voor PRO gedlt dat dit een links/progressieve partij is, die vrij ver van de CDA ligt (en nog verder van VVD). 

Voor de fracties met veel verworpen zaken geldt nogmaals het FVD, maar deze wordt nu opgevolgd door de PVV en de PvdD. Zowel FVD en PVV zijn rechts/conservatief, maar de PvdD is links/progressief. Ook is opmerkelijk dat de PvdD ook voorkomt in de aangehouden categorie, met hetzelfde aantal zaken. Hieruit is te concluderen dat deze partijen vaak te extreem zijn, waardoor hun zaken te ver bij de rest van de fracties weg liggen. Hierdoor worden de meeste van hun zaken dus verworpen.

# 3. Welke soorten besluiten (aangenomen, verworpen, aangehouden) komen het meeste voor per soort zaak?

In [ ]:
# Maakt een nieuwe kolom met alleen trefwoord voor makelijkere analyse
besluit["soort_match"] = besluit["besluit_soort"].str.extract(
    r"(aangenomen|verworpen|aangehouden)",
    expand=False)

# Dataframe dat bij deze vraag gebruikt wordt
besluiten_per_zaak = query_df("""
    SELECT z.soort AS soort_zaak, b.soort_match AS besluit
    FROM zaak z
    JOIN besluit_zaak bz
        ON z.id = bz.zaak_id
    JOIN besluit b
        ON bz.besluit_id = b.id""")

In [ ]:
# Berekent percentages van besluit per soort zaak
percentage_per_zaak = (
    besluiten_per_zaak
    .dropna(subset=["besluit"])
    .groupby(["soort_zaak", "besluit"])
    .size()
    .reset_index(name="aantal zaken")
)

# Maakt de presentatie netjes
percentage_per_zaak["percentage"] = (
    percentage_per_zaak["aantal zaken"]
    / percentage_per_zaak.groupby("soort_zaak")["aantal zaken"].transform("sum")
    * 100
).round(2)

# Selecteerd alleen de hoogste percentages
# Per soort zaak dus alleen het besluit dat het meeste voorkomt
meest_voorkomende = (
    percentage_per_zaak
    .sort_values(["soort_zaak", "percentage"], ascending=[True, False])
    .drop_duplicates("soort_zaak")
)

meest_voorkomende

Voor bijna alle soort zaken geldt dat het meestvoorkomende soort besluit 'aangenomen' is. Opvallend is dat wetsvoorstel, begroting, brief kamer, en wijzigingen door regering allen een 100 procent aangenomen percentage hebben. Alleen verzoek werkzaamheden heeft een 100 procent verworpen. 

Ongeacht het besluit, zijn de percentages bij alle soort zaken vrij hoog. De enige met een redelijk even verdeling over de drie besluiten (aangehouden/verworpen/aangenomen) is motie. 

# 4. Aantal activiteiten op één dag

## 4.1 Aantal eendaagse activiteiten op één dag
Hiervoor kunnen we regelrecht de data uit het silver tabel 'activiteit' gebruiken. 

In [ ]:
# Selecteren van de eendaagse activiteiten
activiteit_per_dag = (
    activiteit[activiteit["datum_soort"] == "Dag"]
    .groupby("datum")
    .size()
    .reset_index(name="aantal_activiteiten")
)

In [ ]:
activiteit_per_dag["aantal_activiteiten"].describe()

In [ ]:
# Upper bound voor leesbaarheid x-as
upper_bound = activiteit_per_dag["aantal_activiteiten"].quantile(0.90)

(
    activiteit_per_dag["aantal_activiteiten"]
    .loc[lambda x: x <= upper_bound]
    .value_counts()
    .sort_index()
    .plot.bar(
        xlabel="Aantal activiteiten op een dag",
        ylabel="Frequentie",
        title="Aantal dagen per aantal activiteiten"
    )
)

Gemiddeld zijn er ongeveer 16 activiteiten op een dag. Vanuit de barplot is echter duidelijk te zien dat er vaak maximaal 6 activiteiten op een dag zijn, met de mediaan op 9 activiteiten per dag. Het gemiddelde wordt zwaar omhoog getrokken door de uitzonderingen: 95% van de tijd zijn er minder dan 38 activiteiten op een dag, maar het maximum van 155 veroorzaakt veel verstoring. Hierom is de barplot ook afgekapt bij 39 activiteiten. 

## 4.2 Gaan de eendaagse activiteiten die op dezelfde dag besproken worden vaak over hetzelfde onderwerp? 

In [ ]:
# Activiteiten worden alleen geteld als ze over andere onderwerpen gaan
per_activiteit_per_dag = (
    activiteit[activiteit["datum_soort"] == "Dag"]
    .groupby("datum")["soort"]
    .nunique()
    .reset_index(name="aantal_verschillende_soorten")
)

In [ ]:
per_activiteit_per_dag["aantal_verschillende_soorten"].value_counts().sort_index().plot.bar(
    xlabel="Aantal verschillende activiteitstypen op een dag",
    ylabel="Frequentie",
    title="Aantal dagen pet aantal activiteitstypen")

Dit grafiek geeft duidelijk een ander beeld dan de algemene activiteiten op 1 dag. Hoewel er veel activiteiten behandeld kunnen worden op een dag, zullen een groot deel vaak over hetzelfde onderwerp gaan. 